# 03a — NTSB Data Acquisition (Problem C)

Download and parse the NTSB aviation accident/incident database.

**Data source:** NTSB `avall.zip` (Microsoft Access `.mdb` format)  
**Scope:** 2018–2020 (POC)  
**Output:** `data/raw/ntsb_accidents.parquet`

In [3]:
import os
from pathlib import Path

# Find project root (contains pyproject.toml)
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [4]:
import requests
import zipfile
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# POC date range
YEAR_START = 2018
YEAR_END   = 2020
print(f'POC scope: {YEAR_START}–{YEAR_END}')

POC scope: 2018–2020


## 1. Download NTSB Data

The NTSB provides bulk data via `avall.zip` (Access `.mdb` format).  
On Windows we use `pyodbc` with the MS Access ODBC driver to read it.

**Alternative:** If pyodbc/Access driver isn't available, we fall back to downloading
pre-converted CSV from the NTSB CAROL portal.

In [5]:
MDB_PATH = RAW_DIR / 'avall.mdb'
ZIP_URL  = 'https://data.ntsb.gov/avdata/FileDirectory/DownloadFile?fileID=C%3A%5Cavdata%5Cavall.zip'

if not MDB_PATH.exists():
    print('Downloading NTSB avall.zip (~120 MB)...')
    resp = requests.get(ZIP_URL, stream=True, timeout=300)
    resp.raise_for_status()
    
    zip_path = RAW_DIR / 'avall.zip'
    with open(zip_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'Downloaded {zip_path.stat().st_size / 1e6:.1f} MB')
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_DIR)
        print(f'Extracted: {z.namelist()}')
else:
    print(f'NTSB data already exists at {MDB_PATH}')

NTSB data already exists at data\raw\avall.mdb


## 2. Read MDB with pyodbc

In [6]:
import pyodbc

# Build connection string for MS Access
conn_str = (
    r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
    f'DBQ={MDB_PATH.resolve()};'
)

try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    
    # List all tables in the database
    tables = [row.table_name for row in cursor.tables(tableType='TABLE')]
    print(f'Tables in avall.mdb ({len(tables)}):')
    for t in tables:
        cursor.execute(f'SELECT COUNT(*) FROM [{t}]')
        count = cursor.fetchone()[0]
        print(f'  {t}: {count:,} rows')
    
    USE_PYODBC = True
except Exception as e:
    print(f'pyodbc connection failed: {e}')
    print('Will use fallback CSV approach.')
    USE_PYODBC = False

Tables in avall.mdb (20):
  aircraft: 30,877 rows
  Country: 262 rows
  ct_iaids: 955 rows
  ct_seqevt: 2,224 rows
  dt_aircraft: 262,723 rows
  dt_events: 113,728 rows
  dt_Flight_Crew: 174,307 rows
  eADMSPUB_DataDictionary: 4,574 rows
  engines: 27,848 rows
  events: 30,358 rows
  Events_Sequence: 64,907 rows
  Findings: 71,114 rows
  Flight_Crew: 31,667 rows
  flight_time: 398,263 rows
  injury: 174,550 rows
  narratives: 27,877 rows
  NTSB_Admin: 30,358 rows
  Occurrences: 0 rows
  seq_of_events: 0 rows
  states: 51 rows


In [7]:
if USE_PYODBC:
    # Read the main 'events' table
    events = pd.read_sql('SELECT * FROM [events]', conn)
    print(f'Events table: {events.shape}')
    
    # Read aircraft table for aircraft details
    if 'aircraft' in [t.lower() for t in tables]:
        aircraft_table = [t for t in tables if t.lower() == 'aircraft'][0]
        aircraft = pd.read_sql(f'SELECT * FROM [{aircraft_table}]', conn)
        print(f'Aircraft table: {aircraft.shape}')
    else:
        aircraft = None
        print('No aircraft table found')
    
    conn.close()
    print('\nEvents columns:')
    print(events.columns.tolist())
else:
    print('Skipping pyodbc read — see fallback cell below.')

C:\Users\yashn\AppData\Local\Temp\ipykernel_12908\3263349865.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events = pd.read_sql('SELECT * FROM [events]', conn)


Events table: (30358, 73)


C:\Users\yashn\AppData\Local\Temp\ipykernel_12908\3263349865.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  aircraft = pd.read_sql(f'SELECT * FROM [{aircraft_table}]', conn)


Aircraft table: (30877, 93)

Events columns:
['ev_id', 'ntsb_no', 'ev_type', 'ev_date', 'ev_dow', 'ev_time', 'ev_tmzn', 'ev_city', 'ev_state', 'ev_country', 'ev_site_zipcode', 'ev_year', 'ev_month', 'mid_air', 'on_ground_collision', 'latitude', 'longitude', 'latlong_acq', 'apt_name', 'ev_nr_apt_id', 'ev_nr_apt_loc', 'apt_dist', 'apt_dir', 'apt_elev', 'wx_brief_comp', 'wx_src_iic', 'wx_obs_time', 'wx_obs_dir', 'wx_obs_fac_id', 'wx_obs_elev', 'wx_obs_dist', 'wx_obs_tmzn', 'light_cond', 'sky_cond_nonceil', 'sky_nonceil_ht', 'sky_ceil_ht', 'sky_cond_ceil', 'vis_rvr', 'vis_rvv', 'vis_sm', 'wx_temp', 'wx_dew_pt', 'wind_dir_deg', 'wind_dir_ind', 'wind_vel_kts', 'wind_vel_ind', 'gust_ind', 'gust_kts', 'altimeter', 'wx_dens_alt', 'wx_int_precip', 'metar', 'ev_highest_injury', 'inj_f_grnd', 'inj_m_grnd', 'inj_s_grnd', 'inj_tot_f', 'inj_tot_m', 'inj_tot_n', 'inj_tot_s', 'inj_tot_t', 'invest_agy', 'ntsb_docket', 'ntsb_notf_from', 'ntsb_notf_date', 'ntsb_notf_tm', 'fiche_number', 'lchg_date', 'lchg

## 2b. Fallback: Manual CSV Download

If pyodbc fails, download CSV from NTSB CAROL:
1. Go to https://my.ntsb.gov/aviation
2. Set date range 2018-01-01 to 2020-12-31
3. Click **Download Summary (CSV)**
4. Save as `data/raw/ntsb_carol_export.csv`

Then run the cell below:

In [8]:
CAROL_CSV = RAW_DIR / 'ntsb_carol_export.csv'

if not USE_PYODBC and CAROL_CSV.exists():
    events = pd.read_csv(CAROL_CSV, low_memory=False)
    aircraft = None  # CAROL export is denormalized
    print(f'Loaded CAROL CSV: {events.shape}')
    print(events.columns.tolist())
elif not USE_PYODBC:
    print(f'ERROR: Neither pyodbc nor {CAROL_CSV} available.')
    print('Please download the CAROL CSV manually (see instructions above).')

## 3. Parse and Clean NTSB Data

In [9]:
# Standardize column names (NTSB uses inconsistent casing)
events.columns = [c.strip().lower() for c in events.columns]
print(f'Columns: {events.columns.tolist()}')

# Identify key fields — NTSB schema uses these names
# (exact names depend on MDB vs CAROL export)
DATE_COL = next((c for c in events.columns if 'ev_date' in c or 'event_date' in c or 'eventdate' in c), None)
AIRPORT_COL = next((c for c in events.columns if 'arpt_id' in c or 'airport' in c), None)
STATE_COL = next((c for c in events.columns if 'ev_state' in c or 'state' in c), None)
CITY_COL = next((c for c in events.columns if 'ev_city' in c or 'city' in c), None)
INJURY_COL = next((c for c in events.columns if 'inj_tot_f' in c or 'highest_injury' in c or 'highestinjurylevel' in c), None)
EVID_COL = next((c for c in events.columns if 'ev_id' in c or 'ntsb_no' in c or 'ntsbno' in c), None)

print(f'\nDetected key columns:')
print(f'  Date:    {DATE_COL}')
print(f'  Airport: {AIRPORT_COL}')
print(f'  State:   {STATE_COL}')
print(f'  City:    {CITY_COL}')
print(f'  Injury:  {INJURY_COL}')
print(f'  EventID: {EVID_COL}')

Columns: ['ev_id', 'ntsb_no', 'ev_type', 'ev_date', 'ev_dow', 'ev_time', 'ev_tmzn', 'ev_city', 'ev_state', 'ev_country', 'ev_site_zipcode', 'ev_year', 'ev_month', 'mid_air', 'on_ground_collision', 'latitude', 'longitude', 'latlong_acq', 'apt_name', 'ev_nr_apt_id', 'ev_nr_apt_loc', 'apt_dist', 'apt_dir', 'apt_elev', 'wx_brief_comp', 'wx_src_iic', 'wx_obs_time', 'wx_obs_dir', 'wx_obs_fac_id', 'wx_obs_elev', 'wx_obs_dist', 'wx_obs_tmzn', 'light_cond', 'sky_cond_nonceil', 'sky_nonceil_ht', 'sky_ceil_ht', 'sky_cond_ceil', 'vis_rvr', 'vis_rvv', 'vis_sm', 'wx_temp', 'wx_dew_pt', 'wind_dir_deg', 'wind_dir_ind', 'wind_vel_kts', 'wind_vel_ind', 'gust_ind', 'gust_kts', 'altimeter', 'wx_dens_alt', 'wx_int_precip', 'metar', 'ev_highest_injury', 'inj_f_grnd', 'inj_m_grnd', 'inj_s_grnd', 'inj_tot_f', 'inj_tot_m', 'inj_tot_n', 'inj_tot_s', 'inj_tot_t', 'invest_agy', 'ntsb_docket', 'ntsb_notf_from', 'ntsb_notf_date', 'ntsb_notf_tm', 'fiche_number', 'lchg_date', 'lchg_userid', 'wx_cond_basic', 'faa_dist

In [10]:
# Parse dates and filter to POC range
if DATE_COL:
    events['event_date'] = pd.to_datetime(events[DATE_COL], errors='coerce')
    events['year'] = events['event_date'].dt.year
    
    print(f'Full date range: {events["event_date"].min()} to {events["event_date"].max()}')
    print(f'Total events: {len(events):,}')
    
    # Filter to POC range
    mask = events['year'].between(YEAR_START, YEAR_END)
    ntsb = events[mask].copy()
    print(f'\nFiltered to {YEAR_START}-{YEAR_END}: {len(ntsb):,} events')
    print(f'Events per year:')
    print(ntsb['year'].value_counts().sort_index())
else:
    print('ERROR: Could not find date column. Manual mapping needed.')
    print(f'Available columns: {events.columns.tolist()}')

Full date range: 2008-01-01 00:00:00 to 2026-03-30 00:00:00
Total events: 30,358

Filtered to 2018-2020: 4,704 events
Events per year:
year
2018    1685
2019    1624
2020    1395
Name: count, dtype: int64


## 4. Join Aircraft Details (if available)

In [11]:
if aircraft is not None:
    aircraft.columns = [c.strip().lower() for c in aircraft.columns]
    
    # Find join key
    join_key = next((c for c in aircraft.columns if 'ev_id' in c), None)
    if join_key and EVID_COL:
        # Take first aircraft per event (primary)
        aircraft_first = aircraft.drop_duplicates(subset=[join_key], keep='first')
        ntsb = ntsb.merge(aircraft_first, left_on=EVID_COL, right_on=join_key, 
                          how='left', suffixes=('', '_acft'))
        print(f'Joined aircraft data. Shape: {ntsb.shape}')
    else:
        print(f'Could not find join key. Aircraft columns: {aircraft.columns.tolist()}')
else:
    print('No aircraft table to join (CAROL export is already denormalized).')

Joined aircraft data. Shape: (4704, 167)


## 5. Inspect & Profile

In [12]:
print(f'Dataset shape: {ntsb.shape}')
print(f'\nColumn info:')
for col in ntsb.columns:
    non_null = ntsb[col].notna().sum()
    pct = non_null / len(ntsb) * 100
    nunique = ntsb[col].nunique()
    print(f'  {col:40s} {pct:5.1f}% fill  {nunique:6d} unique')

Dataset shape: (4704, 167)

Column info:
  ev_id                                    100.0% fill    4704 unique
  ntsb_no                                  100.0% fill    4704 unique
  ev_type                                  100.0% fill       2 unique
  ev_date                                  100.0% fill    1077 unique
  ev_dow                                   100.0% fill       7 unique
  ev_time                                   98.9% fill     949 unique
  ev_tmzn                                   95.5% fill       1 unique
  ev_city                                  100.0% fill    2944 unique
  ev_state                                  81.4% fill      55 unique
  ev_country                               100.0% fill     117 unique
  ev_site_zipcode                           77.0% fill    2526 unique
  ev_year                                  100.0% fill       3 unique
  ev_month                                 100.0% fill      12 unique
  mid_air                                    1.4%

In [13]:
# Airport coverage — how many events have usable airport codes?
if AIRPORT_COL:
    has_airport = ntsb[AIRPORT_COL].notna() & (ntsb[AIRPORT_COL].str.strip() != '')
    print(f'Events with airport code: {has_airport.sum():,} ({has_airport.mean()*100:.1f}%)')
    print(f'\nTop 20 airports:')
    print(ntsb[has_airport][AIRPORT_COL].value_counts().head(20))
    
    # Monthly distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ntsb['year'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Events per Year')
    ntsb['event_date'].dt.month.value_counts().sort_index().plot(kind='bar', ax=axes[1], title='Events per Month')
    axes[1].set_xlabel('Month')
    plt.tight_layout()
    plt.show()

## 6. Save Cleaned NTSB Data

In [14]:
out_path = RAW_DIR / 'ntsb_accidents.parquet'
ntsb.to_parquet(out_path, index=False)
print(f'Saved {len(ntsb):,} NTSB events to {out_path}')
print(f'File size: {out_path.stat().st_size / 1e6:.1f} MB')

Saved 4,704 NTSB events to data\raw\ntsb_accidents.parquet
File size: 1.2 MB
